# NOTEBOOK MODELO

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [2]:
df_no_caucasian = pd.read_csv(r"C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_no_caucasian.csv")
df_no_african = pd.read_csv(r"C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_no_african.csv")
df_estudio = pd.read_csv(r'C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_no_african.csv')

In [3]:
df_no_caucasian.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'priors_count', 'juv_priors_count', 'c_charge_degree', 'maritalstatus',
       'maritalstatus_other', 'maritalstatus_separated',
       'maritalstatus_significant other', 'race_african-american',
       'race_hispanic', 'race_other'],
      dtype='object')

In [26]:
lista_columnas_dropear = ['two_year_recid','person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid', 'is_violent_recid', 'maritalstatus', 'race', 'maritalstatus_other', 'maritalstatus_separated',
       'maritalstatus_significant other', 'juv_priors_count']       #revisar una vez esté todo claro

## Modelo No Caucasian

In [27]:
X1 = df_no_caucasian.drop(columns=lista_columnas_dropear)
y1 = df_no_caucasian['two_year_recid']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size = 0.2, random_state = 42, stratify = y1)

model1 = LogisticRegression(max_iter=1000)

model1.fit(X1_train, y1_train)

y1_pred = model1.predict(X1_test)
y1_proba = model1.predict_proba(X1_test)[:,1]

print("Matriz de confusión:")
print(confusion_matrix(y1_test, y1_pred))
print("\nReporte de clasificación:")
print(classification_report(y1_test, y1_pred))
print("ROC AUC:", roc_auc_score(y1_test, y1_proba))

Matriz de confusión:
[[719  65]
 [291 153]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.71      0.92      0.80       784
           1       0.70      0.34      0.46       444

    accuracy                           0.71      1228
   macro avg       0.71      0.63      0.63      1228
weighted avg       0.71      0.71      0.68      1228

ROC AUC: 0.7176310558006986


## Modelo No African

In [28]:
X2 = df_no_african.drop(columns=lista_columnas_dropear)
y2 = df_no_african['two_year_recid']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size = 0.2, random_state = 42, stratify = y2)

model2 = LogisticRegression(max_iter=2000)

model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)
y2_proba = model2.predict_proba(X2_test)[:,1]

print("Matriz de confusión:")
print(confusion_matrix(y2_test, y2_pred))
print("\nReporte de clasificación:")
print(classification_report(y2_test, y2_pred))
print("ROC AUC:", roc_auc_score(y2_test, y2_proba))

Matriz de confusión:
[[719  65]
 [291 153]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.71      0.92      0.80       784
           1       0.70      0.34      0.46       444

    accuracy                           0.71      1228
   macro avg       0.71      0.63      0.63      1228
weighted avg       0.71      0.71      0.68      1228

ROC AUC: 0.7176712745909174


In [40]:
y2_test.value_counts(normalize=True)

two_year_recid
0    0.638436
1    0.361564
Name: proportion, dtype: float64

In [41]:
y2_train.value_counts(normalize=True)

two_year_recid
0    0.638419
1    0.361581
Name: proportion, dtype: float64

In [43]:
df_test.groupby('race')['two_year_recid'].value_counts(normalize=True)

race              two_year_recid
african-american  0                 0.579278
                  1                 0.420722
caucasian         0                 0.697337
                  1                 0.302663
hispanic          0                 0.788991
                  1                 0.211009
other             0                 0.594203
                  1                 0.405797
Name: proportion, dtype: float64

## SESGO

In [42]:
df_test = df_no_african.loc[X2_test.index].copy()

df_test['y_true'] = y2_test
df_test['y_pred'] = y2_pred
df_test['y_proba'] = y2_proba

df_test_caucasian = df_test[df_test['race'] == 'caucasian']
df_test_african = df_test[df_test['race'] == 'african-american']

print("CAUCASIAN")
print(confusion_matrix(df_test_caucasian['y_true'], df_test_caucasian['y_pred'], normalize= 'true'))
print(classification_report(df_test_caucasian['y_true'], df_test_caucasian['y_pred']))

print("AFRICAN-AMERICAN")
print(confusion_matrix(df_test_african['y_true'], df_test_african['y_pred'], normalize= 'true'))
print(classification_report(df_test_african['y_true'], df_test_african['y_pred']))

CAUCASIAN
[[0.96180556 0.03819444]
 [0.768      0.232     ]]
              precision    recall  f1-score   support

           0       0.74      0.96      0.84       288
           1       0.72      0.23      0.35       125

    accuracy                           0.74       413
   macro avg       0.73      0.60      0.59       413
weighted avg       0.74      0.74      0.69       413

AFRICAN-AMERICAN
[[0.86449864 0.13550136]
 [0.55970149 0.44029851]]
              precision    recall  f1-score   support

           0       0.68      0.86      0.76       369
           1       0.70      0.44      0.54       268

    accuracy                           0.69       637
   macro avg       0.69      0.65      0.65       637
weighted avg       0.69      0.69      0.67       637



In [29]:
#Mirar FP Y FN de este modelo según Edad

In [ ]:
df_test = df_no_african.loc[X2_test.index].copy()

df_test['y_true'] = y2_test
df_test['y_pred'] = y2_pred
df_test['y_proba'] = y2_proba

df_test_less_25 = df_test[df_test['race'] == 'caucasian']
df_test_25_45 = df_test[df_test['race'] == 'african-american']
df_test_46_65 = df_test[df_test['race'] == 'african-american']

print("Less than 25")
print(confusion_matrix(df_test_caucasian['y_true'], df_test_caucasian['y_pred'], normalize= 'true'))
print(classification_report(df_test_caucasian['y_true'], df_test_caucasian['y_pred']))

print("25-45")
print(confusion_matrix(df_test_african['y_true'], df_test_african['y_pred'], normalize= 'true'))
print(classification_report(df_test_african['y_true'], df_test_african['y_pred']))

print("46-65")
print(confusion_matrix(df_test_african['y_true'], df_test_african['y_pred'], normalize= 'true'))
print(classification_report(df_test_african['y_true'], df_test_african['y_pred']))

## Nuestro Modelo

In [ ]:
lista_columnas_dropear = ['two_year_recid','person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid', 'is_violent_recid', 'maritalstatus', 'race', 'maritalstatus_other', 'maritalstatus_separated',
       'maritalstatus_significant other', 'juv_priors_count']       #revisar una vez esté todo claro

In [39]:
X = df_no_caucasian.drop(columns=lista_columnas_dropear)
y = df_no_caucasian['two_year_recid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

model = LogisticRegression(max_iter=000)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:,1]
y_pred = (y_proba >= 0.5).astype(int)

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

Matriz de confusión:
[[784   0]
 [444   0]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.64      1.00      0.78       784
           1       0.00      0.00      0.00       444

    accuracy                           0.64      1228
   macro avg       0.32      0.50      0.39      1228
weighted avg       0.41      0.64      0.50      1228

ROC AUC: 0.5945644305019304
